# 📘 Notebook 9: GRPO - Group Relative Policy Optimization
## Building LLMs from Scratch Workshop

> **GRPO is the algorithm behind DeepSeek-R1** - one of the most capable open-source reasoning models. It is simpler than PPO, needs no critic model, and achieves strong alignment results.

---

### Why GRPO After PPO?

In Notebook 6, we implemented PPO-based RLHF. PPO works, but has significant drawbacks:

| Issue with PPO | How GRPO Fixes It |
|----------------|-------------------|
| Needs a **separate critic/value model** (extra memory + training) | **No critic model** - advantages computed from group statistics |
| Training is **unstable** (sensitive to hyperparameters) | **More stable** - relative advantages are naturally normalized |
| Requires **3-4 models in memory** (policy, reference, reward, critic) | Only **2 models** needed (policy + reference) + reward |
| Complex implementation (GAE, value loss, entropy bonus) | **Much simpler** - single clean loss function |

### The Key Insight of GRPO

For each prompt, generate **G responses** (a "group"). Score them all with the reward model. Then compute **relative advantages within the group**:

$$\hat{A}_i = \frac{r_i - \text{mean}(\mathbf{r})}{\text{std}(\mathbf{r})}$$

Responses that are **better than average** in their group get positive advantage. Worse than average get negative. This is elegant because:
- No value function needed to estimate baselines
- The group itself IS the baseline
- Naturally handles different difficulty levels across prompts

### GRPO Loss Function

$$\mathcal{L}_{\text{GRPO}} = -\frac{1}{G} \sum_{i=1}^{G} \min\left(\frac{\pi_\theta(o_i|q)}{\pi_{\text{old}}(o_i|q)} \hat{A}_i, \; \text{clip}\left(\frac{\pi_\theta(o_i|q)}{\pi_{\text{old}}(o_i|q)}, 1-\epsilon, 1+\epsilon\right) \hat{A}_i \right) - \beta \cdot D_{\text{KL}}(\pi_\theta \| \pi_{\text{ref}})$$

Where:
- $G$ = group size (number of responses per prompt)
- $\hat{A}_i$ = group-relative advantage for response $i$
- $\epsilon$ = clipping parameter (same as PPO)
- $\beta$ = KL penalty coefficient
- $\pi_{\text{ref}}$ = frozen reference model (SFT checkpoint)

---

### What You Will Build

1. **Full GRPO algorithm** from scratch in PyTorch
2. **Group sampling** - generate multiple responses per prompt
3. **Group-relative advantage** computation
4. **Clipped policy gradient** with KL penalty
5. **Side-by-side comparison**: Base vs SFT vs PPO vs GRPO

---


## 1. Setup

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import math, time, json, re, copy, os
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


Device: cpu


In [2]:
# ============================================================
# Project Directory Setup
# ============================================================
if os.path.basename(os.getcwd()) == 'notebooks':
    PROJECT_ROOT = os.path.abspath('..')
else:
    PROJECT_ROOT = os.path.abspath('.')

DATA_DIR    = os.path.join(PROJECT_ROOT, 'data')
MODELS_DIR  = os.path.join(PROJECT_ROOT, 'models')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR}")
print(f"Models dir:   {MODELS_DIR}")
print(f"Results dir:  {RESULTS_DIR}")


Project root: d:\Documents\Unisole\UniTransformerWorkshop
Data dir:     d:\Documents\Unisole\UniTransformerWorkshop\data
Models dir:   d:\Documents\Unisole\UniTransformerWorkshop\models
Results dir:  d:\Documents\Unisole\UniTransformerWorkshop\results


## 2. Model Components (from Previous Notebooks)

In [3]:
# ============================================================
# Tokenizer + GPT Model + Reward Model (compact)
# ============================================================
class BPETokenizer:
    def __init__(self, nm=100):
        self.num_merges=nm; self.merges={}; self.vocab={}; self.inverse_vocab={}
        self.special_tokens={'<PAD>':0,'<UNK>':1,'<BOS>':2,'<EOS>':3}
    def _gwf(self, text):
        return {tuple(list(w)+['</w>']):f for w,f in Counter(re.findall(r'\S+',text.lower())).items()}
    def _gpc(self, wf):
        p=Counter()
        for wt,f in wf.items():
            for i in range(len(wt)-1): p[(wt[i],wt[i+1])]+=f
        return p
    def _mp(self, wf, pair):
        m=pair[0]+pair[1]; nwf={}
        for wt,f in wf.items():
            nt=[]; i=0
            while i<len(wt):
                if i<len(wt)-1 and wt[i]==pair[0] and wt[i+1]==pair[1]: nt.append(m); i+=2
                else: nt.append(wt[i]); i+=1
            nwf[tuple(nt)]=f
        return nwf
    def train(self, text):
        wf=self._gwf(text); ic=set(t for wt in wf for t in wt)
        for i in range(self.num_merges):
            ps=self._gpc(wf)
            if not ps: break
            bp,c=ps.most_common(1)[0]
            if c<2: break
            self.merges[bp]=bp[0]+bp[1]; wf=self._mp(wf,bp)
        self.vocab=dict(self.special_tokens); idx=len(self.special_tokens)
        for ch in sorted(ic):
            if ch not in self.vocab: self.vocab[ch]=idx; idx+=1
        for p,m in self.merges.items():
            if m not in self.vocab: self.vocab[m]=idx; idx+=1
        self.inverse_vocab={v:k for k,v in self.vocab.items()}
        print(f"Tokenizer: vocab={len(self.vocab)}")
    def _tw(self, word):
        t=list(word)+['</w>']
        for p,m in self.merges.items():
            i=0
            while i<len(t)-1:
                if t[i]==p[0] and t[i+1]==p[1]: t=t[:i]+[m]+t[i+2:]
                else: i+=1
        return t
    def encode(self, text):
        return [self.vocab.get(t,1) for w in re.findall(r'\S+',text.lower()) for t in self._tw(w)]
    def decode(self, ids):
        return ''.join(self.inverse_vocab.get(i,'?') for i in ids).replace('</w>',' ').strip()
    @property
    def vocab_size(self): return len(self.vocab)

class LayerNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__(); self.eps=eps
        self.gamma=nn.Parameter(torch.ones(d)); self.beta=nn.Parameter(torch.zeros(d))
    def forward(self, x):
        return self.gamma*(x-x.mean(-1,keepdim=True))/torch.sqrt(x.var(-1,keepdim=True,unbiased=False)+self.eps)+self.beta

class CausalMHA(nn.Module):
    def __init__(self, cfg):
        super().__init__(); d=cfg['emb_dim']; self.nh=cfg['num_heads']; self.hd=d//self.nh
        self.qkv=nn.Linear(d,3*d,bias=False); self.out=nn.Linear(d,d,bias=False)
        self.ad=nn.Dropout(cfg['drop_rate']); self.rd=nn.Dropout(cfg['drop_rate'])
        self.register_buffer('mask',torch.triu(torch.ones(cfg['context_length'],cfg['context_length']),diagonal=1).bool())
    def forward(self, x):
        B,T,C=x.shape; Q,K,V=self.qkv(x).chunk(3,dim=-1)
        Q=Q.view(B,T,self.nh,self.hd).transpose(1,2); K=K.view(B,T,self.nh,self.hd).transpose(1,2)
        V=V.view(B,T,self.nh,self.hd).transpose(1,2)
        a=Q@K.transpose(-2,-1)/(self.hd**0.5)
        a.masked_fill_(self.mask[:T,:T].unsqueeze(0).unsqueeze(0),float('-inf'))
        return self.rd(self.out((self.ad(F.softmax(a,dim=-1))@V).transpose(1,2).contiguous().view(B,T,C)))

class FFN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(cfg['emb_dim'],cfg['emb_dim']*cfg.get('ff_mult',4)),
            nn.GELU(),nn.Linear(cfg['emb_dim']*cfg.get('ff_mult',4),cfg['emb_dim']),nn.Dropout(cfg['drop_rate']))
    def forward(self, x): return self.net(x)

class TBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1=LayerNorm(cfg['emb_dim']); self.attn=CausalMHA(cfg)
        self.ln2=LayerNorm(cfg['emb_dim']); self.ffn=FFN(cfg)
    def forward(self, x): x=x+self.attn(self.ln1(x)); return x+self.ffn(self.ln2(x))

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__(); self.config=cfg
        self.tok_emb=nn.Embedding(cfg['vocab_size'],cfg['emb_dim'])
        self.pos_emb=nn.Embedding(cfg['context_length'],cfg['emb_dim'])
        self.drop=nn.Dropout(cfg['drop_rate'])
        self.blocks=nn.Sequential(*[TBlock(cfg) for _ in range(cfg['num_layers'])])
        self.final_ln=LayerNorm(cfg['emb_dim'])
        self.lm_head=nn.Linear(cfg['emb_dim'],cfg['vocab_size'],bias=False)
        self.lm_head.weight=self.tok_emb.weight
        self.apply(self._iw)
    def _iw(self, m):
        if isinstance(m,nn.Linear):
            nn.init.normal_(m.weight,std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m,nn.Embedding): nn.init.normal_(m.weight,std=0.02)
    def forward(self, ids, targets=None):
        B,T=ids.shape
        x=self.drop(self.tok_emb(ids)+self.pos_emb(torch.arange(T,device=ids.device)))
        x=self.lm_head(self.final_ln(self.blocks(x)))
        loss=None
        if targets is not None: loss=F.cross_entropy(x.view(-1,x.size(-1)),targets.view(-1))
        return x, loss
    def get_hidden_states(self, ids):
        B,T=ids.shape
        x=self.drop(self.tok_emb(ids)+self.pos_emb(torch.arange(T,device=ids.device)))
        return self.final_ln(self.blocks(x))

class RewardModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone=GPTModel(cfg)
        self.value_head=nn.Sequential(
            nn.Linear(cfg['emb_dim'],cfg['emb_dim']//2),nn.GELU(),nn.Linear(cfg['emb_dim']//2,1))
        for m in self.value_head:
            if isinstance(m,nn.Linear):
                nn.init.normal_(m.weight,std=0.02); nn.init.zeros_(m.bias)
    def forward(self, ids):
        h=self.backbone.get_hidden_states(ids)
        return self.value_head(h[:,-1,:])

print("All model components loaded!")


All model components loaded!


## 3. Initialize Tokenizer, Models, and Data

In [4]:
# ============================================================
# Build tokenizer + training corpus
# ============================================================
CORPUS = """Machine learning is a subset of artificial intelligence.
Neural networks learn representations from data automatically.
Transformers revolutionized natural language processing.
Attention is all you need was the groundbreaking paper.
Large language models can generate human-like text.
Training requires massive amounts of text data.
Tokenization converts text into numerical sequences.
Self-attention allows tokens to attend to all others.
Fine-tuning adapts pretrained models to specific tasks.
RLHF uses human preferences to align model outputs.
A reward model scores responses based on preferences.
PPO is a policy optimization algorithm for RLHF.
GRPO uses group relative advantages without a critic.
DPO optimizes preferences directly without reward models.
The KL penalty prevents diverging from the reference model.
Gradient clipping prevents exploding gradients during training.
Layer normalization stabilizes deep network training.
Residual connections help gradient flow in deep architectures.
Temperature controls randomness of text generation output.
Top-k sampling limits generation to the most likely tokens.
""" * 10

INST_TEXT = " ".join([
    "### instruction: what is machine learning? ### response: machine learning is a branch of AI that enables systems to learn patterns from data without explicit programming.",
    "### instruction: explain attention. ### response: attention lets each token compute relevance scores with all other tokens using queries keys and values.",
    "### instruction: what is RLHF? ### response: RLHF aligns models using human preference feedback through a reward model and policy optimization.",
    "### instruction: what is GRPO? ### response: GRPO is group relative policy optimization that computes advantages from group statistics without needing a critic model.",
    "### instruction: explain tokenization. ### response: tokenization converts raw text into numerical tokens that the model can process using methods like BPE.",
    "### instruction: what is a transformer? ### response: a transformer is a neural network architecture that uses self-attention to process sequences in parallel.",
    "### instruction: how does gradient descent work? ### response: gradient descent iteratively updates parameters by moving in the direction that reduces the loss function.",
    "### instruction: explain the reward model. ### response: a reward model is trained on human preferences to score model outputs and guide policy optimization.",
] * 10)

tokenizer = BPETokenizer(nm=250)
tokenizer.train(CORPUS + " " + INST_TEXT)

GPT_CONFIG = {
    "vocab_size": tokenizer.vocab_size,
    "context_length": 64,
    "emb_dim": 128,
    "num_heads": 4,
    "num_layers": 4,
    "drop_rate": 0.1,
    "ff_mult": 4,
}

print(f"Config: vocab={GPT_CONFIG['vocab_size']}, emb={GPT_CONFIG['emb_dim']}, layers={GPT_CONFIG['num_layers']}")


Tokenizer: vocab=285
Config: vocab=285, emb=128, layers=4


In [5]:
# ============================================================
# Train a Reward Model (same as Notebook 6)
# ============================================================
PREFERENCE_DATA = [
    {"prompt": "what is machine learning?",
     "chosen": "machine learning is a branch of artificial intelligence that enables systems to learn and improve from experience and data without being explicitly programmed using algorithms to find patterns.",
     "rejected": "machine learning is something computers do. it is related to technology and stuff."},
    {"prompt": "explain neural networks.",
     "chosen": "a neural network is a computing system inspired by biological brains consisting of layers of interconnected nodes that process information through weighted connections and activation functions.",
     "rejected": "neural networks are like brains but for computers. they do AI things."},
    {"prompt": "what is attention in transformers?",
     "chosen": "attention is a mechanism that allows each token in a sequence to compute relevance scores with all other tokens using queries keys and values to create weighted combinations.",
     "rejected": "attention is a way for the model to look at words. it helps with understanding."},
    {"prompt": "how does gradient descent work?",
     "chosen": "gradient descent is an optimization algorithm that iteratively updates model parameters by computing the gradient of the loss function and moving in the direction that reduces loss.",
     "rejected": "gradient descent makes the model better by changing numbers using math."},
    {"prompt": "what is RLHF?",
     "chosen": "RLHF stands for reinforcement learning from human feedback. it trains a reward model on human preference data and uses that reward signal to optimize the language model policy.",
     "rejected": "RLHF is a training method that makes AI better using feedback from people."},
    {"prompt": "what is GRPO?",
     "chosen": "GRPO is group relative policy optimization which generates multiple responses per prompt and uses within-group reward statistics to compute advantages without needing a critic model.",
     "rejected": "GRPO is an algorithm for training language models. it makes them better."},
    {"prompt": "explain tokenization.",
     "chosen": "tokenization converts raw text into smaller units called tokens which are mapped to numerical IDs. methods include byte-pair encoding which iteratively merges frequent character pairs.",
     "rejected": "tokenization splits text into pieces. the model uses these pieces."},
    {"prompt": "what is transfer learning?",
     "chosen": "transfer learning applies knowledge gained from training on one task to a different but related task. in NLP pretraining on large corpora then fine-tuning is a powerful form of transfer learning.",
     "rejected": "transfer learning means using a model for different things. it saves time."},
] * 6

# Build preference dataset
class PreferenceDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=64):
        self.samples = []
        pad_id = tokenizer.vocab.get('<PAD>', 0)
        for item in data:
            p = item['prompt']
            c_ids = tokenizer.encode(f"{p} {item['chosen']}")[:max_length]
            r_ids = tokenizer.encode(f"{p} {item['rejected']}")[:max_length]
            c_ids += [pad_id] * (max_length - len(c_ids))
            r_ids += [pad_id] * (max_length - len(r_ids))
            self.samples.append({
                'chosen_ids': torch.tensor(c_ids[:max_length], dtype=torch.long),
                'rejected_ids': torch.tensor(r_ids[:max_length], dtype=torch.long),
            })
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        return s['chosen_ids'], s['rejected_ids']

pref_dataset = PreferenceDataset(PREFERENCE_DATA, tokenizer, GPT_CONFIG['context_length'])
pref_loader = DataLoader(pref_dataset, batch_size=4, shuffle=True, drop_last=True)

# Train reward model
reward_model = RewardModel(GPT_CONFIG).to(device)
rm_opt = torch.optim.AdamW(reward_model.parameters(), lr=1e-4, weight_decay=0.01)

print("Training Reward Model...")
for epoch in range(20):
    reward_model.train()
    total_loss = 0; total_correct = 0; total_n = 0
    for c_ids, r_ids in pref_loader:
        c_ids, r_ids = c_ids.to(device), r_ids.to(device)
        rc = reward_model(c_ids)
        rr = reward_model(r_ids)
        loss = -F.logsigmoid(rc - rr).mean()
        rm_opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(reward_model.parameters(), 1.0)
        rm_opt.step()
        total_loss += loss.item()
        with torch.no_grad():
            total_correct += (rc > rr).float().sum().item()
            total_n += c_ids.size(0)
    if (epoch+1) % 5 == 0:
        print(f"  Epoch {epoch+1:2d} | Loss: {total_loss/len(pref_loader):.4f} | Acc: {total_correct/total_n:.1%}")

# Freeze reward model
for p in reward_model.parameters(): p.requires_grad = False
reward_model.eval()
print("Reward model trained and frozen!")


Training Reward Model...
  Epoch  5 | Loss: 0.2019 | Acc: 100.0%
  Epoch 10 | Loss: 0.0413 | Acc: 100.0%
  Epoch 15 | Loss: 0.0136 | Acc: 100.0%
  Epoch 20 | Loss: 0.0069 | Acc: 100.0%
Reward model trained and frozen!


In [6]:
# ============================================================
# Initialize policy, reference, and (optional) PPO baseline
# ============================================================

# Policy model (will be optimized by GRPO)
policy_model = GPTModel(GPT_CONFIG).to(device)

# Reference model (frozen SFT checkpoint)
ref_model = GPTModel(GPT_CONFIG).to(device)
ref_model.load_state_dict(policy_model.state_dict())
for p in ref_model.parameters(): p.requires_grad = False
ref_model.eval()

# Save a copy for PPO comparison later
ppo_baseline_model = GPTModel(GPT_CONFIG).to(device)
ppo_baseline_model.load_state_dict(policy_model.state_dict())

print(f"Policy model:    {sum(p.numel() for p in policy_model.parameters() if p.requires_grad):,} trainable params")
print(f"Reference model: frozen")
print(f"Reward model:    frozen")
print(f"\nGRPO needs: Policy + Reference + Reward = 3 models")
print(f"PPO needs:  Policy + Reference + Reward + Critic = 4 models")
print(f"GRPO saves memory by eliminating the critic entirely!")


Policy model:    835,968 trainable params
Reference model: frozen
Reward model:    frozen

GRPO needs: Policy + Reference + Reward = 3 models
PPO needs:  Policy + Reference + Reward + Critic = 4 models
GRPO saves memory by eliminating the critic entirely!


---
## 4. GRPO Core Algorithm - Step by Step

### The GRPO Training Loop

```
For each prompt q:
    1. Generate G responses:  o_1, o_2, ..., o_G  ~ pi_old(.|q)
    2. Score each response:   r_1, r_2, ..., r_G  = RM(q, o_i)
    3. Compute group-relative advantages:
            A_i = (r_i - mean(r)) / std(r)
    4. For each response, compute the clipped policy loss:
            ratio = pi_theta(o_i|q) / pi_old(o_i|q)
            L_i = min(ratio * A_i, clip(ratio, 1-eps, 1+eps) * A_i)
    5. Add KL penalty:
            KL = per-token KL divergence from reference
    6. Total loss = -mean(L_i) + beta * KL
    7. Update policy theta
```

### Key Difference from PPO

| Component | PPO | GRPO |
|-----------|-----|------|
| Advantage estimation | GAE using a **learned value function** | **Group-relative normalization** (no value function) |
| Models in memory | Policy + Ref + Reward + **Critic** | Policy + Ref + Reward |
| Baseline | Learned value baseline | Group mean as baseline |
| Advantage formula | $A_t = \delta_t + \gamma\lambda\delta_{t+1} + ...$ | $\hat{A}_i = \frac{r_i - \bar{r}}{\sigma_r}$ |


In [7]:
# ============================================================
# GRPO Helper Functions
# ============================================================

@torch.no_grad()
def generate_group_responses(model, prompt_ids, group_size=4, max_new_tokens=20,
                              temperature=0.8, top_k=20):
    '''
    Generate G (group_size) different responses for a single prompt.
    
    This is the core of GRPO: multiple responses per prompt enable
    relative advantage computation without a critic.
    
    Args:
        model: Policy model
        prompt_ids: [1, prompt_len] tokenized prompt
        group_size: G - number of responses to generate per prompt
        max_new_tokens: Tokens to generate per response
        temperature: Sampling temperature
        top_k: Top-k filtering
    
    Returns:
        full_sequences: [G, prompt_len + max_new_tokens] all full sequences
        response_tokens: [G, max_new_tokens] just the generated parts
    '''
    model.eval()
    ctx = model.config['context_length']
    
    # Repeat the prompt G times
    prompt_repeated = prompt_ids.repeat(group_size, 1)  # [G, prompt_len]
    generated = prompt_repeated.clone()
    
    for _ in range(max_new_tokens):
        idx = generated[:, -ctx:]
        logits, _ = model(idx)
        logits = logits[:, -1, :] / max(temperature, 1e-8)
        
        if top_k:
            tv, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < tv[:, -1:]] = float('-inf')
        
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, 1)  # [G, 1]
        generated = torch.cat([generated, next_token], dim=1)
    
    response_tokens = generated[:, prompt_ids.shape[1]:]
    return generated, response_tokens


def compute_sequence_log_probs(model, full_ids, prompt_len):
    '''
    Compute per-token log probabilities for the RESPONSE portion only.
    
    Args:
        model: Language model
        full_ids: [B, T] full sequence (prompt + response)
        prompt_len: Length of the prompt (we ignore this part)
    
    Returns:
        log_probs: [B] sum of log-probs over response tokens
        per_token_log_probs: [B, response_len] per-token log-probs
    '''
    logits, _ = model(full_ids[:, :-1])  # [B, T-1, V]
    log_probs = F.log_softmax(logits, dim=-1)
    
    # Gather log probs for actual tokens
    targets = full_ids[:, 1:]  # [B, T-1]
    token_lp = log_probs.gather(2, targets.unsqueeze(-1)).squeeze(-1)  # [B, T-1]
    
    # Only sum over response tokens (after prompt_len - 1 because of shift)
    response_start = max(0, prompt_len - 1)
    response_lp = token_lp[:, response_start:]  # [B, response_len]
    
    return response_lp.sum(dim=-1), response_lp  # [B], [B, response_len]


def compute_group_advantages(rewards, eps=1e-8):
    '''
    Compute group-relative advantages.
    
    This is the KEY innovation of GRPO:
    Instead of using a learned value function (critic) as baseline,
    use the group mean as the baseline.
    
    A_i = (r_i - mean(r)) / (std(r) + eps)
    
    Args:
        rewards: [G] reward scores for each response in the group
    
    Returns:
        advantages: [G] normalized advantages
    '''
    mean_r = rewards.mean()
    std_r = rewards.std()
    
    # Normalize: responses better than group average get positive advantage
    advantages = (rewards - mean_r) / (std_r + eps)
    
    return advantages


def compute_per_token_kl(policy_logits, ref_logits):
    '''
    Compute per-token KL divergence: KL(policy || ref).
    
    Using the exact formula:
    KL = sum_x p(x) * log(p(x)/q(x))
    
    DeepSeek-R1 uses an unbiased estimator:
    KL = exp(log_ref - log_policy) - (log_ref - log_policy) - 1
    
    Args:
        policy_logits: [B, T, V] logits from policy
        ref_logits: [B, T, V] logits from reference
    
    Returns:
        kl: [B, T] per-token KL divergence
    '''
    policy_log_probs = F.log_softmax(policy_logits, dim=-1)
    ref_log_probs = F.log_softmax(ref_logits, dim=-1)
    
    # KL divergence per position
    # Using the DeepSeek formulation for stability:
    # KL_i = exp(ref_log_p - policy_log_p) - (ref_log_p - policy_log_p) - 1
    log_ratio = ref_log_probs - policy_log_probs
    kl = (torch.exp(log_ratio) - log_ratio - 1)
    
    # Sum over vocabulary dimension
    policy_probs = F.softmax(policy_logits, dim=-1)
    kl = (policy_probs * kl).sum(dim=-1)  # [B, T]
    
    return kl


print("GRPO helper functions defined!")
print()
print("Key functions:")
print("  generate_group_responses()  - Sample G responses per prompt")
print("  compute_sequence_log_probs() - Get log-probs for response tokens")
print("  compute_group_advantages()  - THE key GRPO innovation")
print("  compute_per_token_kl()      - KL penalty from reference")


GRPO helper functions defined!

Key functions:
  generate_group_responses()  - Sample G responses per prompt
  compute_sequence_log_probs() - Get log-probs for response tokens
  compute_group_advantages()  - THE key GRPO innovation
  compute_per_token_kl()      - KL penalty from reference


---
## 5. The Complete GRPO Training Loop

Now we put it all together. For each training iteration:

1. **Sample a prompt** from our pool
2. **Generate G responses** using the current policy
3. **Score all G responses** with the reward model
4. **Compute group-relative advantages** (no critic needed!)
5. **Compute clipped policy gradient loss** + KL penalty
6. **Update the policy**

This is substantially simpler than PPO because we never need to:
- Train a value/critic model
- Compute GAE (Generalized Advantage Estimation)
- Balance value loss + policy loss


In [8]:
# ============================================================
# GRPO Training Prompts
# ============================================================
GRPO_PROMPTS = [
    "what is machine learning?",
    "explain neural networks.",
    "what is attention?",
    "how does gradient descent work?",
    "what is RLHF?",
    "explain tokenization.",
    "what is transfer learning?",
    "explain cross-entropy loss.",
    "what is a transformer?",
    "how does text generation work?",
    "what is perplexity?",
    "explain GRPO algorithm.",
]

prompt_prefix = "### instruction: "
response_prefix = " ### response: "

# Tokenize all prompts
tokenized_prompts = []
for p in GRPO_PROMPTS:
    ids = tokenizer.encode(prompt_prefix + p + response_prefix)
    tokenized_prompts.append(ids)

# Pad to uniform length
max_prompt_len = min(max(len(p) for p in tokenized_prompts), GPT_CONFIG['context_length'] // 2)
padded_prompts = []
for ids in tokenized_prompts:
    ids = ids[:max_prompt_len]
    ids = ids + [0] * (max_prompt_len - len(ids))
    padded_prompts.append(ids)

prompt_tensor = torch.tensor(padded_prompts, dtype=torch.long).to(device)
print(f"Prompts: {len(GRPO_PROMPTS)}, padded to {max_prompt_len} tokens")


Prompts: 12, padded to 18 tokens


In [9]:
# ============================================================
# GRPO TRAINING LOOP
# ============================================================

# Hyperparameters
GRPO_EPOCHS      = 30      # Number of outer iterations
GROUP_SIZE       = 4       # G: responses per prompt (core GRPO parameter!)
MAX_NEW_TOKENS   = 15      # Response length
CLIP_EPSILON     = 0.2     # PPO-style clipping
KL_COEFF         = 0.05    # Beta: KL penalty strength
GRPO_LR          = 5e-5    # Learning rate
TEMPERATURE      = 0.8     # Generation temperature
PPO_INNER_ITERS  = 2       # Inner optimization steps per batch

# Optimizer
grpo_optimizer = torch.optim.AdamW(policy_model.parameters(), lr=GRPO_LR, weight_decay=0.01)

# Tracking
grpo_rewards = []
grpo_kl_divs = []
grpo_losses = []
grpo_advantages_std = []  # Track advantage distribution

print("=" * 65)
print("  GRPO TRAINING (Group Relative Policy Optimization)")
print("=" * 65)
print(f"  Group size G={GROUP_SIZE} | Clip={CLIP_EPSILON} | KL={KL_COEFF} | LR={GRPO_LR}")
print(f"  Max new tokens={MAX_NEW_TOKENS} | Epochs={GRPO_EPOCHS}")
print("=" * 65)

start_time = time.time()
ctx_len = GPT_CONFIG['context_length']

for epoch in range(GRPO_EPOCHS):
    epoch_reward = 0
    epoch_kl = 0
    epoch_loss = 0
    num_prompts_processed = 0
    
    # Shuffle prompts each epoch
    perm = torch.randperm(prompt_tensor.size(0))
    
    for prompt_idx in range(len(perm)):
        idx = perm[prompt_idx]
        prompt_ids = prompt_tensor[idx:idx+1]  # [1, prompt_len]
        prompt_len = prompt_ids.shape[1]
        
        # ─────────────────────────────────────────────
        # STEP 1: Generate G responses from current policy
        # ─────────────────────────────────────────────
        policy_model.eval()
        full_seqs, resp_tokens = generate_group_responses(
            policy_model, prompt_ids,
            group_size=GROUP_SIZE,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_k=20
        )
        # full_seqs: [G, prompt_len + max_new_tokens]
        
        # Truncate to context length
        if full_seqs.shape[1] > ctx_len:
            full_seqs = full_seqs[:, :ctx_len]
        
        # ─────────────────────────────────────────────
        # STEP 2: Score all G responses with reward model
        # ─────────────────────────────────────────────
        pad_id = 0
        rm_input = full_seqs
        if rm_input.shape[1] < ctx_len:
            rm_input = F.pad(rm_input, (0, ctx_len - rm_input.shape[1]), value=pad_id)
        else:
            rm_input = rm_input[:, :ctx_len]
        
        with torch.no_grad():
            rewards = reward_model(rm_input).squeeze(-1)  # [G]
        
        # ─────────────────────────────────────────────
        # STEP 3: Compute GROUP-RELATIVE advantages
        # ─────────────────────────────────────────────
        # THIS IS THE KEY GRPO INNOVATION!
        advantages = compute_group_advantages(rewards)  # [G]
        
        # ─────────────────────────────────────────────
        # STEP 4: Compute old log probs (before update)
        # ─────────────────────────────────────────────
        policy_model.eval()
        with torch.no_grad():
            old_seq_lp, old_token_lp = compute_sequence_log_probs(
                policy_model, full_seqs, prompt_len
            )  # old_seq_lp: [G]
        
        # ─────────────────────────────────────────────
        # STEP 5: PPO-style clipped update with KL
        # ─────────────────────────────────────────────
        policy_model.train()
        
        for inner_iter in range(PPO_INNER_ITERS):
            # Current log probs
            cur_seq_lp, cur_token_lp = compute_sequence_log_probs(
                policy_model, full_seqs, prompt_len
            )
            
            # Probability ratio
            ratio = torch.exp(cur_seq_lp - old_seq_lp)  # [G]
            
            # Clipped surrogate loss
            surr1 = ratio * advantages
            surr2 = torch.clamp(ratio, 1 - CLIP_EPSILON, 1 + CLIP_EPSILON) * advantages
            policy_loss = -torch.min(surr1, surr2).mean()
            
            # KL penalty (per-token)
            policy_logits, _ = policy_model(full_seqs[:, :-1])
            with torch.no_grad():
                ref_logits, _ = ref_model(full_seqs[:, :-1])
            
            per_token_kl = compute_per_token_kl(policy_logits, ref_logits)
            # Only penalize KL on response tokens
            response_kl = per_token_kl[:, max(0, prompt_len-1):].mean()
            
            # Total GRPO loss
            total_loss = policy_loss + KL_COEFF * response_kl
            
            grpo_optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(policy_model.parameters(), 1.0)
            grpo_optimizer.step()
        
        # Track metrics
        epoch_reward += rewards.mean().item()
        epoch_kl += response_kl.item()
        epoch_loss += total_loss.item()
        num_prompts_processed += 1
    
    # Epoch averages
    n = max(num_prompts_processed, 1)
    avg_reward = epoch_reward / n
    avg_kl = epoch_kl / n
    avg_loss = epoch_loss / n
    
    grpo_rewards.append(avg_reward)
    grpo_kl_divs.append(avg_kl)
    grpo_losses.append(avg_loss)
    
    if (epoch + 1) % 5 == 0:
        elapsed = time.time() - start_time
        print(f"  Epoch {epoch+1:3d} | Reward: {avg_reward:+.4f} | "
              f"KL: {avg_kl:.4f} | Loss: {avg_loss:.4f} | Time: {elapsed:.0f}s")

total_time = time.time() - start_time
print(f"\nGRPO training complete! Time: {total_time:.0f}s")


  GRPO TRAINING (Group Relative Policy Optimization)
  Group size G=4 | Clip=0.2 | KL=0.05 | LR=5e-05
  Max new tokens=15 | Epochs=30
  Epoch   5 | Reward: -2.7827 | KL: 0.1885 | Loss: 0.0499 | Time: 13s
  Epoch  10 | Reward: -2.7768 | KL: 0.7558 | Loss: 0.0857 | Time: 27s
  Epoch  15 | Reward: -2.7738 | KL: 2.0137 | Loss: 0.1251 | Time: 41s
  Epoch  20 | Reward: -2.7728 | KL: 3.0246 | Loss: 0.1618 | Time: 56s
  Epoch  25 | Reward: -2.7726 | KL: 3.1515 | Loss: 0.1598 | Time: 73s
  Epoch  30 | Reward: -2.7726 | KL: 3.2807 | Loss: 0.1714 | Time: 89s

GRPO training complete! Time: 89s


---
## 6. PPO Baseline (for Fair Comparison)

We train PPO on the same prompts, same reward model, same number of epochs.


In [10]:
# ============================================================
# Quick PPO training (same setup for fair comparison)
# ============================================================
ppo_model = ppo_baseline_model  # Start from same init
ppo_ref = GPTModel(GPT_CONFIG).to(device)
ppo_ref.load_state_dict(ppo_model.state_dict())
for p in ppo_ref.parameters(): p.requires_grad = False
ppo_ref.eval()

ppo_optimizer = torch.optim.AdamW(ppo_model.parameters(), lr=GRPO_LR, weight_decay=0.01)

ppo_rewards_track = []
ppo_kl_track = []

print("Training PPO baseline for comparison...")
start_time = time.time()

for epoch in range(GRPO_EPOCHS):
    ppo_model.eval()
    epoch_reward = 0; epoch_kl = 0; n_prompts = 0
    perm = torch.randperm(prompt_tensor.size(0))
    
    for pi in range(len(perm)):
        idx = perm[pi]
        p_ids = prompt_tensor[idx:idx+1]
        p_len = p_ids.shape[1]
        
        # Generate single response (PPO style)
        gen = p_ids.clone()
        for _ in range(MAX_NEW_TOKENS):
            idx_c = gen[:, -ctx_len:]
            logits, _ = ppo_model(idx_c)
            logits = logits[:, -1, :] / TEMPERATURE
            tv, _ = torch.topk(logits, 20)
            logits[logits < tv[:, -1:]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1)
            gen = torch.cat([gen, nxt], 1)
        
        if gen.shape[1] > ctx_len:
            gen = gen[:, :ctx_len]
        
        # Reward
        rm_in = gen
        if rm_in.shape[1] < ctx_len:
            rm_in = F.pad(rm_in, (0, ctx_len - rm_in.shape[1]), value=0)
        with torch.no_grad():
            reward = reward_model(rm_in[:, :ctx_len]).squeeze(-1)
        
        # Log probs
        with torch.no_grad():
            old_lp, _ = compute_sequence_log_probs(ppo_model, gen, p_len)
            ref_lp, _ = compute_sequence_log_probs(ppo_ref, gen, p_len)
        
        kl_val = (old_lp - ref_lp).mean()
        advantage = reward - KL_COEFF * kl_val
        advantage = (advantage - advantage.mean()) / (advantage.std() + 1e-8)
        
        # PPO update
        ppo_model.train()
        for _ in range(PPO_INNER_ITERS):
            cur_lp, _ = compute_sequence_log_probs(ppo_model, gen, p_len)
            ratio = torch.exp(cur_lp - old_lp)
            s1 = ratio * advantage
            s2 = torch.clamp(ratio, 1 - CLIP_EPSILON, 1 + CLIP_EPSILON) * advantage
            loss = -torch.min(s1, s2).mean()
            ppo_optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(ppo_model.parameters(), 1.0)
            ppo_optimizer.step()
        
        epoch_reward += reward.mean().item()
        epoch_kl += kl_val.item()
        n_prompts += 1
    
    ppo_rewards_track.append(epoch_reward / max(n_prompts, 1))
    ppo_kl_track.append(epoch_kl / max(n_prompts, 1))
    
    if (epoch + 1) % 10 == 0:
        print(f"  PPO Epoch {epoch+1:3d} | Reward: {ppo_rewards_track[-1]:+.4f} | KL: {ppo_kl_track[-1]:.4f}")

print(f"PPO baseline done! Time: {time.time()-start_time:.0f}s")


Training PPO baseline for comparison...


C:\Users\disha\AppData\Local\Temp\ipykernel_26568\1768093291.py:57: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1857.)
  advantage = (advantage - advantage.mean()) / (advantage.std() + 1e-8)


RuntimeError: probability tensor contains either `inf`, `nan` or element < 0

---
## 7. GRPO vs PPO: Training Curves


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Reward comparison
axes[0].plot(grpo_rewards, 'g-', linewidth=2, label='GRPO')
axes[0].plot(ppo_rewards_track, 'b--', linewidth=2, label='PPO')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Avg Reward')
axes[0].set_title('Reward: GRPO vs PPO')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# KL comparison
axes[1].plot(grpo_kl_divs, 'g-', linewidth=2, label='GRPO')
axes[1].plot(ppo_kl_track, 'b--', linewidth=2, label='PPO')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('KL Divergence')
axes[1].set_title('KL Divergence from Reference')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# GRPO loss
axes[2].plot(grpo_losses, 'r-', linewidth=2)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('GRPO Loss')
axes[2].set_title('GRPO Surrogate Loss')
axes[2].grid(True, alpha=0.3)

plt.suptitle('GRPO vs PPO Training Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'grpo_vs_ppo.png'), dpi=120, bbox_inches='tight')
plt.show()


---
## 8. Side-by-Side Output Comparison: Reference vs PPO vs GRPO


In [ ]:
@torch.no_grad()
def gen_text(model, tokenizer, prompt, max_new=30, temp=0.7, top_k=15, device='cpu'):
    model.eval()
    ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long).to(device)
    ctx = model.config['context_length']
    for _ in range(max_new):
        idx = ids[:, -ctx:]
        logits, _ = model(idx)
        logits = logits[:, -1, :] / max(temp, 1e-8)
        if top_k:
            tv, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < tv[:, -1:]] = float('-inf')
        probs = F.softmax(logits, dim=-1)
        nxt = torch.multinomial(probs, 1) if temp > 0 else torch.argmax(probs, -1, keepdim=True)
        ids = torch.cat([ids, nxt], 1)
    return tokenizer.decode(ids[0].tolist())

def extract_response(text):
    if "### response:" in text:
        return text.split("### response:")[-1].strip()
    return text.strip()

print("=" * 75)
print("  REFERENCE (SFT)  vs  PPO  vs  GRPO  -  Output Comparison")
print("=" * 75)

test_prompts = [
    "what is machine learning?",
    "explain attention mechanism.",
    "what is GRPO?",
    "how does gradient descent work?",
]

for q in test_prompts:
    prompt = f"### instruction: {q} ### response:"
    print(f"\n{'=' * 70}")
    print(f"  Q: {q}")
    print(f"{'=' * 70}")
    
    for name, model in [("REF ", ref_model), ("PPO ", ppo_model), ("GRPO", policy_model)]:
        out = gen_text(model, tokenizer, prompt, max_new=30, device=device)
        resp = extract_response(out)[:100]
        
        # Score with reward model
        r_ids = tokenizer.encode(out)[:ctx_len]
        r_ids += [0] * (ctx_len - len(r_ids))
        score = reward_model(torch.tensor([r_ids]).to(device)).item()
        
        print(f"  {name} (r={score:+.3f}): {resp}")


---
## 9. Why GRPO Works: Intuition and Analysis

### The Group-Relative Advantage is Brilliant

Consider a prompt with 4 generated responses and their rewards:

```
Response A: reward = 0.8  → advantage = +1.2  (best in group, big positive update)
Response B: reward = 0.3  → advantage = +0.1  (slightly above average)
Response C: reward = 0.1  → advantage = -0.5  (below average, pushed down)
Response D: reward = -0.2 → advantage = -0.8  (worst, strongly pushed down)
```

The model learns to **generate more responses like A and B, fewer like C and D** - all without ever training a value function!

### Why No Critic Is Needed

In PPO, the critic estimates $V(s)$ (expected future reward) to reduce variance. But this requires:
- A separate neural network
- A separate training objective
- Careful balancing of value loss vs policy loss

GRPO achieves variance reduction through **group normalization**:
- Group mean acts as the baseline (like V(s))
- Group std normalizes the scale
- Larger groups = lower variance (but more compute)

### Group Size Trade-offs

| Group Size G | Advantage Quality | Compute Cost | Recommendation |
|-------------|-------------------|-------------|----------------|
| G = 2 | High variance, noisy | Low | Quick experiments |
| G = 4 | Good balance | Medium | **Default choice** |
| G = 8 | Lower variance | High | When quality matters |
| G = 16+ | Very stable | Very high | Large-scale training |

DeepSeek-R1 uses **G = 64** with large models.


In [ ]:
# ============================================================
# Visualize how group-relative advantage works
# ============================================================

# Simulate rewards for 4 different prompts, each with G=8 responses
np.random.seed(42)
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

prompt_names = [
    "Easy prompt (all high reward)",
    "Hard prompt (all low reward)",
    "Mixed prompt (varied)",
    "Bimodal prompt (good vs bad)",
]

reward_sets = [
    np.random.normal(0.7, 0.1, 8),    # Easy: all high
    np.random.normal(0.1, 0.1, 8),    # Hard: all low
    np.random.uniform(-0.2, 0.9, 8),  # Mixed
    np.concatenate([np.random.normal(0.8, 0.05, 4), np.random.normal(0.1, 0.05, 4)]),  # Bimodal
]

for ax, name, rewards in zip(axes.flat, prompt_names, reward_sets):
    # Compute group-relative advantages
    mean_r = rewards.mean()
    std_r = rewards.std() + 1e-8
    advantages = (rewards - mean_r) / std_r
    
    x = np.arange(len(rewards))
    colors = ['#2ecc71' if a > 0 else '#e74c3c' for a in advantages]
    
    # Plot rewards
    bars = ax.bar(x - 0.2, rewards, 0.35, label='Reward', color='steelblue', alpha=0.7)
    # Plot advantages
    bars2 = ax.bar(x + 0.2, advantages, 0.35, label='Advantage', color=colors, alpha=0.8)
    
    ax.axhline(y=mean_r, color='steelblue', linestyle='--', alpha=0.5, label=f'Group mean={mean_r:.2f}')
    ax.axhline(y=0, color='gray', linewidth=0.5)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Response Index')
    ax.set_ylabel('Score')
    ax.legend(fontsize=7, loc='upper right')
    ax.set_xticks(x)
    ax.set_xticklabels([f'R{i}' for i in range(len(rewards))])
    ax.grid(True, alpha=0.2, axis='y')

plt.suptitle('GRPO: Group-Relative Advantages Across Different Prompt Types',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'grpo_advantages.png'), dpi=120, bbox_inches='tight')
plt.show()

print("Key insight from the plots:")
print("  - Easy prompt: even high-reward responses get negative advantage if below group mean")
print("  - Hard prompt: even low-reward responses get positive advantage if above group mean")
print("  - This RELATIVE scoring is what eliminates the need for a learned value function!")


---
## 10. Save GRPO Model


In [ ]:
torch.save({
    'grpo_policy_state_dict': policy_model.state_dict(),
    'ppo_model_state_dict': ppo_model.state_dict(),
    'ref_state_dict': ref_model.state_dict(),
    'reward_state_dict': reward_model.state_dict(),
    'config': GPT_CONFIG,
    'tokenizer_vocab': tokenizer.vocab,
    'tokenizer_merges': {f"{k[0]}|||{k[1]}": v for k, v in tokenizer.merges.items()},
    'tokenizer_special_tokens': tokenizer.special_tokens,
    'tokenizer_num_merges': tokenizer.num_merges,
    'grpo_rewards': grpo_rewards,
    'grpo_kl_divs': grpo_kl_divs,
    'ppo_rewards': ppo_rewards_track,
}, os.path.join(MODELS_DIR, 'grpo_models.pt'))
print(f"Models saved to {os.path.join(MODELS_DIR, 'grpo_models.pt')}")


---
## 11. Exercises

### Exercise 1: Group Size Ablation
Train GRPO with G = 2, 4, 8, 16. Plot reward curves for each. At what G does variance drop noticeably?

### Exercise 2: GRPO vs DPO
Implement DPO (from Notebook 8) on the same preference data. Compare reward curves, training stability, and output quality against GRPO.

### Exercise 3: Rule-Based Rewards (DeepSeek-R1 Style)
DeepSeek-R1 uses **rule-based rewards** (format correctness, answer accuracy) instead of a learned reward model for some tasks. Implement a simple rule-based reward:
```python
def rule_reward(response):
    score = 0.0
    if len(response.split()) > 10: score += 0.3    # Length bonus
    if '?' not in response: score += 0.2            # Not just repeating the question
    if any(kw in response for kw in ['because', 'therefore', 'since']):
        score += 0.5                                 # Reasoning bonus
    return score
```
Train GRPO using this instead of the neural reward model.

### Exercise 4: Outcome vs Process Reward
GRPO can use either **outcome rewards** (score the final response) or **process rewards** (score intermediate reasoning steps). Modify the reward function to give partial credit for each sentence in the response.

---

## 12. Interview Questions

1. **What is the key innovation of GRPO over PPO?**
   - GRPO eliminates the critic/value model by using group-relative advantages. Instead of learning a value function to estimate baselines, it generates multiple responses per prompt and uses the group mean as the baseline.

2. **Why does GRPO need multiple responses per prompt?**
   - The group of responses provides a natural baseline for advantage estimation. A response is "good" if it scores above the group average. Without multiple responses, there is no group to compare against.

3. **How does GRPO compute advantages?**
   - $\hat{A}_i = (r_i - \text{mean}(r_{1:G})) / \text{std}(r_{1:G})$. This normalizes rewards relative to the group, making advantages comparable across prompts with different difficulty levels.

4. **What is the trade-off with group size G?**
   - Larger G gives more stable advantage estimates (lower variance) but costs more compute (G forward passes per prompt). G=4-8 is typical for small models; DeepSeek-R1 uses G=64.

5. **When would you choose GRPO over PPO or DPO?**
   - GRPO over PPO: When memory is limited (no critic needed), when you want simpler implementation, when you have a good reward model.
   - GRPO over DPO: When you want online learning (generate fresh responses each iteration) rather than offline (fixed preference dataset). GRPO can also use non-differentiable reward signals (rule-based, tool outputs).

6. **How does DeepSeek-R1 use GRPO?**
   - DeepSeek-R1 applies GRPO in two stages: (1) GRPO with rule-based rewards on reasoning tasks to develop chain-of-thought, (2) GRPO with a learned reward model for general alignment. The rule-based stage was key to emerging reasoning behavior.

---

## 13. Summary: The Complete Alignment Algorithm Landscape

| Algorithm | Year | Reward Model? | Critic? | Online? | Key Idea |
|-----------|------|--------------|---------|---------|----------|
| **RLHF (PPO)** | 2022 | Yes | Yes | Yes | Learned value baseline, clipped policy gradient |
| **DPO** | 2023 | No | No | No | Implicit reward via preference pairs |
| **GRPO** | 2024 | Yes | **No** | Yes | Group-relative advantages as baseline |
| **KTO** | 2024 | No | No | No | Binary signal (thumbs up/down) |
| **ORPO** | 2024 | No | No | No | SFT + alignment in one step |

**What we built in this workshop:**
- NB 6: PPO (full RLHF with reward model + critic)
- NB 8: DPO (from scratch, in the mini-practicals section)
- **NB 9: GRPO (this notebook!)**

You now have hands-on implementations of the **three most important alignment algorithms** in modern LLM training.

---

*Workshop by Unisole Empower*
